## Assignment 2

In [1]:
#set up
import pandas as pd
from scipy import stats
transaction = pd.read_csv("../data/transaction_data.csv")
transaction

,sku_number,inventory_type,stocking_type,lead_time,unit_price,Manufacturing Site,division Code,transaction_date,Order-Quantity
0,DB45C19C,FG,MTO,28,992.860458,JP-41,NaN,2023-01-01,130
1,83992E02,NaN,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,NaN,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,NaN,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,NaN,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


In [2]:
# rename columns
transaction.columns = ["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price", "manufacturing_site", "division_code", "transaction_date", "order_quantity"]

## 1. Required Transformation

In [9]:
# looking at made to stock items
# focus data for calculating safety stock
mts = transaction.query('stocking_type in "MTS"')
mts

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
1,83992E02,NaN,MTS,21,202.017834,CA-66,ACA9,2023-01-01,39
2,75C364B0,FG,MTS,28,986.009314,JP-41,23F0,2023-01-01,111
3,2C9772C9,NaN,MTS,28,1007.216590,CA-66,9B75,2023-01-01,89
4,ACCCE163,WIP,MTS,28,986.998503,CA-66,6E75,2023-01-01,80
14,D89A6000,FG,MTS,28,988.743492,US-B4,219B,2023-01-01,75
...,...,...,...,...,...,...,...,...,...
784634,F8B91C1B,WIP,MTS,28,998.021118,NaN,23F0,2025-12-30,111
784635,6664F459,FG,MTS,28,99.200335,CA-66,D66F,2025-12-30,1
784636,3DE1D314,FG,MTS,28,1008.523300,NaN,81FE,2025-12-30,115
784637,DB131462,FG,MTS,28,1008.340512,CA-2B,6AF1,2025-12-30,122


In [6]:
mts["order_quantity"].describe()

count    483561.000000
mean         80.052484
std          41.934607
min        -120.000000
25%          64.000000
50%          93.000000
75%         109.000000
max         189.000000
Name: order_quantity, dtype: float64

In [4]:
mts["order_quantity"].min()

np.int64(-120)

In [5]:
mts["order_quantity"].max()

np.int64(189)

In [6]:
mts["order_quantity"].mean()

np.float64(80.05248355429822)

In [7]:
mts["order_quantity"].median()

np.float64(93.0)

In [8]:
mts["order_quantity"].var()

np.float64(1758.511235461733)

In [9]:
mts["order_quantity"].std()

np.float64(41.93460665681428)

In [10]:
mts["lead_time"].mean()

np.float64(25.81905488656033)

## 2. Safety Stock Calculation

In [ ]:
# safety stcok = z-score of the service level coefficient  * demand's standard deviation * sq root of lead time 
# WE NEED
# zscores of what though?
# demands sd = order quanity sd
#


In [ ]:
# calculates z scores at 75%, 90%, and 95%
#* but for each sku? I dont think so, I think this is correct. 
mts_z75 = stats.norm.ppf(.75)
mts_z90 = stats.norm.ppf(.90)
mts_z95 = stats.norm.ppf(.95)

print(mts_z75, mts_z90, mts_z95)

0.6744897501960817 1.2815515655446004 1.6448536269514722


We need 1 row per sku


In [ ]:
# notes: demands sd=order quanity sd so we put std

# average for lead time because its safe? then we can sq root it in equation code

mts_by_sku=(
    mts
    .groupby("sku_number")
    .aggregate({"order_quantity":"std", "lead_time":"mean"})

)
mts_by_sku
# do we need the other features? no

,order_quantity,lead_time
sku_number,,
00234778,10.992029,14.000000
01517B90,20.516397,28.000000
020287B4,20.305351,21.000000
02144443,20.235579,28.000000
0273F5A5,20.689477,21.000000
...,...,...
FD816E1D,19.269821,19.526316
FE482594,20.035266,21.000000
FE564816,10.381967,28.000000


In [ ]:
# safety stock at 75% service level
safety_stock_75 = (
    mts_z75
    *mts_by_sku["order_quantity"]
    *mts_by_sku["lead_time"]** 0.5       #pretty sure we did sq rt this way somehwere in lectures
)
safety_stock_75

sku_number
00234778     27.740688
01517B90     73.224339
020287B4     62.761818
02144443     72.222081
0273F5A5     63.949113
               ...    
FD816E1D     57.433222
FE482594     61.927010
FE564816     37.053906
FEAD0FF0    125.643672
FF6FAC01     99.343299
Length: 509, dtype: float64

In [19]:
# safety stock at 90% service level
safety_stock_90 = (
    mts_z90
    *mts_by_sku["order_quantity"]
    *mts_by_sku["lead_time"]** 0.5       #pretty sure we did sq rt this way somehwere in lectures
)
safety_stock_90

sku_number
00234778     52.708172
01517B90    139.128528
020287B4    119.249411
02144443    137.224207
0273F5A5    121.505309
               ...    
FD816E1D    109.124913
FE482594    117.663250
FE564816     70.403578
FEAD0FF0    238.726896
FF6FAC01    188.755368
Length: 509, dtype: float64

In [20]:
# safety stock at 95% service level
safety_stock_95 = (
    mts_z95
    *mts_by_sku["order_quantity"]
    *mts_by_sku["lead_time"]** 0.5       #pretty sure we did sq rt this way somehwere in lectures
)
safety_stock_95

sku_number
00234778     67.650207
01517B90    178.569533
020287B4    153.054963
02144443    176.125363
0273F5A5    155.950376
               ...    
FD816E1D    140.060310
FE482594    151.019147
FE564816     90.362014
FEAD0FF0    306.402654
FF6FAC01    242.264891
Length: 509, dtype: float64

## 3. Safety Stock Distribution

In [ ]:
#focusing on 95% service level



In [25]:
# largest safety stock
safety_stock_95.sort_values(ascending=False)

sku_number
FCD26C09    380.242570
F39BBC11    380.123486
C195636B    374.462245
9B9EBBD2    371.178003
D6BB12F8    368.513047
               ...    
31CA3F72     53.013323
CD94743C     51.950750
D0405098     51.660810
2462CBBC     51.304287
61D81899     50.393836
Length: 509, dtype: float64

In [26]:
# smallest safety stock
safety_stock_95.sort_values

<bound method Series.sort_values of sku_number
00234778     67.650207
01517B90    178.569533
020287B4    153.054963
02144443    176.125363
0273F5A5    155.950376
               ...    
FD816E1D    140.060310
FE482594    151.019147
FE564816     90.362014
FEAD0FF0    306.402654
FF6FAC01    242.264891
Length: 509, dtype: float64>

In [27]:
safety_stock_95.mean()

np.float64(140.32560664464017)

In [28]:
# add explanation to each section where required


## 4. We should do side quest

# References:

No external references were used in the preparation of this assignment.